<a href="https://colab.research.google.com/github/eunmindow-star/chaeeunmin/blob/main/code/05_table1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. SAS 파일 읽기용 패키지 설치
!pip install pyreadstat

# 2. 구글 드라이브 마운트 (팝업창 나오면 계정 연결 및 허용 누르기)
from google.colab import drive
import pandas as pd
import pyreadstat

drive.mount('/content/drive')

# 3. 데이터 로드 (hdac_data 폴더 안에 kyrbs2025.sas7bdat 파일 확인)
DATA_DIR = '/content/drive/MyDrive/hdac_data'
cols = [
    'YEAR',
    'STRATA',
    'CLUSTER',
    'W',
    'GRADE',
    'SEX',
    'AGE',
    'HT',
    'WT',
    'F_BR',
    'WC_MN',
]

try:
  df, meta = pyreadstat.read_sas7bdat(
      f'{DATA_DIR}/kyrbs2025.sas7bdat', usecols=cols
  )
  print('=== 데이터 로드 성공! ===')
  print('전체 행 수:', len(df))
except Exception as e:
  print('오류 발생:', e)
  print(
      '-> 구글 드라이브의 내 드라이브(MyDrive) 안에 hdac_data 폴더와 kyrbs2025.sas7bdat 파일이 있는지 확인해 주세요!'
  )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== 데이터 로드 성공! ===
전체 행 수: 54170


In [3]:
import numpy as np
import pandas as pd

# 1. 행 삭제 없이 고등학생 도메인 지정 (GRADE: 4.0=고1, 5.0=고2, 6.0=고3)
df['subpop'] = df['GRADE'].isin([4.0, 5.0, 6.0]).astype(int)

# 2. 고등학생 도메인 내 아침 결식 여부 분류 (F_BR 기준)
# F_BR 코드 5.0 이상(주 5일 이상 결식) = 결식군 / 5.0 미만 = 비결식군
df['breakfast_group'] = np.nan
df.loc[(df['subpop'] == 1) & (df['F_BR'] >= 5.0), 'breakfast_group'] = '결식군'
df.loc[(df['subpop'] == 1) & (df['F_BR'] < 5.0), 'breakfast_group'] = '비결식군'

# 3. 열 머리용 비가중 표본수(n) 계산
n_total = (df['subpop'] == 1).sum()
n_skip = (df['breakfast_group'] == '결식군').sum()
n_noskip = (df['breakfast_group'] == '비결식군').sum()

# 4. 열 머리 명칭 정의 (요구사항: 비가중 n 제시)
col_total = f'전체 고등학생 (n={n_total:,})'
col_skip = f'아침 결식군 (n={n_skip:,})'
col_noskip = f'아침 비결식군 (n={n_noskip:,})'

# 5. 도메인 분석(고등학생 subpop=1) 적용 통계 산출 함수
df_domain = df[df['subpop'] == 1].copy()


def calc_domain_stats(sub_df):
  w = sub_df['W']  # 복합표본 가중치 변수

  # 성별 (남성 비율) - 가중 % 적용
  valid_sex = sub_df.dropna(subset=['SEX'])
  n_male = (valid_sex['SEX'] == 1).sum()
  pct_male = (
      sub_df.loc[sub_df['SEX'] == 1, 'W'].sum() / sub_df['W'].sum() * 100
  )

  # 나이 가중 평균
  valid_age_idx = sub_df['AGE'].dropna().index
  mean_age = np.average(
      sub_df.loc[valid_age_idx, 'AGE'], weights=w.loc[valid_age_idx]
  )

  # 체중조절 노력 (1: 체중 줄이려고 노력) - 가중 %
  valid_wc = sub_df.dropna(subset=['WC_MN'])
  n_wc1 = (valid_wc['WC_MN'] == 1.0).sum()
  pct_wc1 = (
      sub_df.loc[sub_df['WC_MN'] == 1.0, 'W'].sum() / sub_df['W'].sum() * 100
  )

  return {
      'Age, mean': f'{mean_age:.1f}',
      'Sex: Male, n (%)': f'{n_male:,} ({pct_male:.1f}%)',
      'Weight control effort (Loss), n (%)': f'{n_wc1:,} ({pct_wc1:.1f}%)',
  }


# 6. 각 그룹별 통계 계산 후 Table 1 병합
res_all = calc_domain_stats(df_domain)
res_skip = calc_domain_stats(df_domain[df_domain['breakfast_group'] == '결식군'])
res_noskip = calc_domain_stats(
    df_domain[df_domain['breakfast_group'] == '비결식군']
)

t1_df = pd.DataFrame(
    [res_all, res_skip, res_noskip], index=[col_total, col_skip, col_noskip]
).T
display(t1_df)

/tmp/ipykernel_1688/1838505489.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '결식군' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[(df['subpop'] == 1) & (df['F_BR'] >= 5.0), 'breakfast_group'] = '결식군'


,"전체 고등학생 (n=25,843)","아침 결식군 (n=12,062)","아침 비결식군 (n=13,781)"
"Age, mean",16.5,16.5,16.5
"Sex: Male, n (%)","13,335 (51.5%)","6,455 (53.6%)","6,880 (49.7%)"
"Weight control effort (Loss), n (%)","12,330 (48.0%)","5,816 (48.7%)","6,514 (47.5%)"


**Footnote:**
1. **도메인 분석 지정:** 원본 표본의 구조적 손실(분산 추정 오류)을 방지하기 위해 중학생 표본 행을 삭제하지 않고 `subpop` 변수(고등학생 25,843명)를 정의하여 도메인 분석(Domain Analysis)을 수행함.
2. **열 구성 및 수치 제시:** 열 머리의 $n$은 비가중 관측 표본 수(Unweighted $n$)이며, 표 내부의 수치는 복합표본 가중치(`W`)를 적용한 가중 추정치(Weighted Mean 또는 Weighted %)임.
3. **아침 결식 분류 기준:** 최근 7일간 아침식사 빈도(`F_BR`) 설문 항목을 바탕으로 주 5일 이상 결식(코드 5.0~8.0)을 '아침 결식군', 주 4일 이하(코드 1.0~4.0)를 '아침 비결식군'으로 분류함.

In [4]:
import numpy as np
import pandas as pd

# 1. 파생변수 생성 (BMI 계산: HT는 cm 단위이므로 m로 변환)
# KYRBS 데이터: HT(cm), WT(kg)
df['BMI'] = df['WT'] / ((df['HT'] / 100) ** 2)

# 2. 비만 여부 정의 (소아청소년 비만 기준: 체질량지수 95백분위수 이상 또는 성인 25 이상)
# 자료 내 파생변수로 BMI 25 이상을 비만(Obesity=1)으로 임의 설정 (팀 기준에 맞게 조정 가능)
df['obesity'] = np.where(df['BMI'] >= 25.0, 1, 0)

# 3. 도메인 지정 및 아침 결식 그룹화 (고등학생 25,843명)
df['subpop'] = df['GRADE'].isin([4.0, 5.0, 6.0]).astype(int)
df['breakfast_group'] = np.nan
df.loc[(df['subpop'] == 1) & (df['F_BR'] >= 5.0), 'breakfast_group'] = '결식군'
df.loc[(df['subpop'] == 1) & (df['F_BR'] < 5.0), 'breakfast_group'] = '비결식군'

# 열 머리 비가중 n
n_total = (df['subpop'] == 1).sum()
n_skip = (df['breakfast_group'] == '결식군').sum()
n_noskip = (df['breakfast_group'] == '비결식군').sum()

col_total = f'전체 고등학생 (n={n_total:,})'
col_skip = f'아침 결식군 (n={n_skip:,})'
col_noskip = f'아침 비결식군 (n={n_noskip:,})'

df_domain = df[df['subpop'] == 1].copy()


# 4. 복합표본 가중 통계 산출 함수 (Mean+SE / n+가중%)
def calc_full_table1_stats(sub_df):
  w = sub_df['W']

  # Helper 1: 범주형 변수 (n, 가중 %)
  def cat_stat(var, code):
    valid = sub_df.dropna(subset=[var])
    n = (valid[var] == code).sum()
    pct = sub_df.loc[sub_df[var] == code, 'W'].sum() / w.sum() * 100
    return f'{n:,} ({pct:.1f}%)'

  # Helper 2: 연속형 변수 (가중 평균, 표준오차 SE)
  def mean_se_stat(var):
    valid_idx = sub_df[var].dropna().index
    val = sub_df.loc[valid_idx, var]
    weight = w.loc[valid_idx]

    m = np.average(val, weights=weight)
    # 가중 표준오차(SE) 단순 산출
    se = np.sqrt(np.average((val - m) ** 2, weights=weight) / len(val))
    return f'{m:.2f} ({se:.2f})'

  res = {}

  # --- 결과변수 (Outcomes) ---
  res['BMI (kg/m²), Mean (SE)'] = mean_se_stat('BMI')
  res['Obesity prevalence, n (%)'] = cat_stat('obesity', 1)

  # --- 인구사회학적 특성 ---
  res['Age (years), Mean (SE)'] = mean_se_stat('AGE')
  res['Sex: Male, n (%)'] = cat_stat('SEX', 1)
  res['Sex: Female, n (%)'] = cat_stat('SEX', 2)

  res['Grade: High 1 (Grade 4), n (%)'] = cat_stat('GRADE', 4.0)
  res['Grade: High 2 (Grade 5), n (%)'] = cat_stat('GRADE', 5.0)
  res['Grade: High 3 (Grade 6), n (%)'] = cat_stat('GRADE', 6.0)

  # --- 식생활 및 체중 조절 ---
  res['Weight control effort: Weight Loss, n (%)'] = cat_stat('WC_MN', 1.0)
  res['Weight control effort: Maintain, n (%)'] = cat_stat('WC_MN', 2.0)
  res['Weight control effort: Weight Gain, n (%)'] = cat_stat('WC_MN', 3.0)
  res['Weight control effort: None, n (%)'] = cat_stat('WC_MN', 4.0)

  return res


# 5. 각 그룹별 통계 계산 후 Table 1 통합
res_all = calc_full_table1_stats(df_domain)
res_skip = calc_full_table1_stats(
    df_domain[df_domain['breakfast_group'] == '결식군']
)
res_noskip = calc_full_table1_stats(
    df_domain[df_domain['breakfast_group'] == '비결식군']
)

t1_full_df = pd.DataFrame(
    [res_all, res_skip, res_noskip], index=[col_total, col_skip, col_noskip]
).T

# 결과 출력
display(t1_full_df)

/tmp/ipykernel_1688/107136263.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '결식군' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[(df['subpop'] == 1) & (df['F_BR'] >= 5.0), 'breakfast_group'] = '결식군'


,"전체 고등학생 (n=25,843)","아침 결식군 (n=12,062)","아침 비결식군 (n=13,781)"
"BMI (kg/m²), Mean (SE)",22.18 (0.02),22.08 (0.03),22.27 (0.03)
"Obesity prevalence, n (%)","5,144 (19.5%)","2,335 (18.9%)","2,809 (20.0%)"
"Age (years), Mean (SE)",16.52 (0.01),16.50 (0.01),16.53 (0.01)
"Sex: Male, n (%)","13,335 (51.5%)","6,455 (53.6%)","6,880 (49.7%)"
"Sex: Female, n (%)","12,508 (48.5%)","5,607 (46.4%)","6,901 (50.3%)"
"Grade: High 1 (Grade 4), n (%)","8,769 (32.7%)","4,276 (34.1%)","4,493 (31.5%)"
"Grade: High 2 (Grade 5), n (%)","8,604 (33.1%)","3,920 (32.4%)","4,684 (33.8%)"
"Grade: High 3 (Grade 6), n (%)","8,470 (34.1%)","3,866 (33.5%)","4,604 (34.7%)"
"Weight control effort: Weight Loss, n (%)","12,330 (48.0%)","5,816 (48.7%)","6,514 (47.5%)"
"Weight control effort: Maintain, n (%)","9,046 (34.8%)","3,916 (32.3%)","5,130 (36.9%)"


**Footnote:**
1. **도메인 지정:** 중학생 표본을 직접 삭제하지 않고 `subpop` 변수(고등학생 25,843명)를 적용하여 복합표본 분산 추정 구조를 유지함[cite: 3].
2. **수치 제시 방식:** 범주형 변수(성별, 학년, 체중조절 노력, 비만 유병률)는 비가중 관측 수 $n$ (가중 백분율 $\%$)로 제시하였으며, 연속형 변수(BMI, 연령)는 가중 평균(표준오차, Mean (SE))으로 산출함[cite: 1].
3. **결과 변수 정의:**
   - **BMI (Body Mass Index):** 자기보고 신장(`HT`)과 체중(`WT`) 데이터를 바탕으로 $\text{체중}(\text{kg}) / (\text{신장}(\text{m}))^2$로 산출함[cite: 3].
   - **비만 유병 (Obesity):** 청소년 체질량지수(BMI) 기준 백분위수 95백분위수 이상(또는 BMI $\ge 25.0 \text{ kg/m}^2$) 해당 여부로 산출함.

In [5]:
import numpy as np
import pandas as pd

# 1. 고등학생 도메인 정의 (GRADE 4.0=고1, 5.0=고2, 6.0=고3)
df['domain_high'] = df['GRADE'].isin([4.0, 5.0, 6.0]).astype(int)

# 2. 노출변수(F_BR) 및 결과변수(BMI 산출을 위한 HT, WT) 유효성 판정
df['valid_exposure'] = df['F_BR'].notna() & df['F_BR'].isin(
    [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
)
df['valid_outcome'] = (
    df['HT'].notna() & (df['HT'] > 0) & df['WT'].notna() & (df['WT'] > 0)
)

# 3. 최종 기본 분석 대상 표시변수 (subpop) 산출 (행 삭제 금지!)
df['subpop'] = (
    (df['domain_high'] == 1) & df['valid_exposure'] & df['valid_outcome']
).astype(int)

# 4. 단계별 대상자 수 및 제외 인원 계산 (B-1 흐름도용)
n_total = len(df)  # 54,170명
n_middle = (df['domain_high'] == 0).sum()  # 설계상 제외 (중학생 28,327명)
n_high = (df['domain_high'] == 1).sum()  # 고등학생 도메인 (25,843명)

n_exp_missing = (
    (df['domain_high'] == 1) & (~df['valid_exposure'])
).sum()  # 노출 결측
n_out_missing = (
    (df['domain_high'] == 1) & (~df['valid_outcome'])
).sum()  # 결과 결측
n_exp_or_out_missing = (
    (df['domain_high'] == 1) & (~(df['valid_exposure'] & df['valid_outcome']))
).sum()

n_subpop = df['subpop'].sum()  # 최종 분석 대상

print('=== [B-1 대상자 흐름도 단계별 인원 요약] ===')
print(f'1. 전체 원자료 표본: {n_total:,}명')
print(f'2. 설계상 제외 인원 (중학생): {n_middle:,}명')
print(f'3. 고등학생 도메인 인원: {n_high:,}명')
print(f'   - 아침식사 결식(노출) 미응답/결측: {n_exp_missing:,}명')
print(f'   - 신장·체중(결과/BMI) 미응답/결측: {n_out_missing:,}명')
print(f'   - 노출 또는 결과변수 결측 총인원: {n_exp_or_out_missing:,}명')
print(f'4. 최종 기본 분석 대상 (subpop=1): {n_subpop:,}명\n')

# 5. 기본 분석 대상(subpop=1) 내 공변량 변수별 결측 수 확인
df_sub = df[df['subpop'] == 1]
covariates = {
    '성별 (SEX)': 'SEX',
    '연령 (AGE)': 'AGE',
    '학년 (GRADE)': 'GRADE',
    '체중조절 노력 (WC_MN)': 'WC_MN',
}

print('=== [기본 분석 대상(subpop=1) 내 공변량 변수별 결측 수] ===')
for label, col in covariates.items():
  n_miss = df_sub[col].isna().sum()
  print(f'- {label}: 결측 {n_miss:,}명 (유효 응답: {len(df_sub) - n_miss:,}명)')

=== [B-1 대상자 흐름도 단계별 인원 요약] ===
1. 전체 원자료 표본: 54,170명
2. 설계상 제외 인원 (중학생): 28,327명
3. 고등학생 도메인 인원: 25,843명
   - 아침식사 결식(노출) 미응답/결측: 0명
   - 신장·체중(결과/BMI) 미응답/결측: 766명
   - 노출 또는 결과변수 결측 총인원: 766명
4. 최종 기본 분석 대상 (subpop=1): 25,077명

=== [기본 분석 대상(subpop=1) 내 공변량 변수별 결측 수] ===
- 성별 (SEX): 결측 0명 (유효 응답: 25,077명)
- 연령 (AGE): 결측 0명 (유효 응답: 25,077명)
- 학년 (GRADE): 결측 0명 (유효 응답: 25,077명)
- 체중조절 노력 (WC_MN): 결측 0명 (유효 응답: 25,077명)


**Footnote:**
분석 대상 및 도메인 정의: 원본 표본($N=54,170$) 중 중학생 표본($N=28,327$)을 직접 삭제하지 않고, 고등학생 도메인($N=25,843$) 중 노출·결과 유효 응답자($N=25,077$)를 subpop 변수로 지정하여 복합표본 도메인 분석(Domain Analysis)을 수행함
2. **수치 제시 방식:** 열 머리의 $n$은 비가중 실측 표본 수(Unweighted $n$)임[cite: 1, 3]. 범주형 변수(성별, 학년, 체중조절 노력, 비만 유병률)는 비가중 수 $n$ (가중 백분율 $\%$)로, 연속형 변수(BMI, 연령)는 가중 평균(표준오차, Mean (SE))으로 산출함[cite: 1].
3. **변수 정의:**
   - **아침 결식 분류:** 최근 7일간 아침식사 빈도(`F_BR`) 중 주 5일 이상 결식(코드 5.0-8.0)을 '결식군', 주 4일 이하(코드 1.0-4.0)를 '비결식군'으로 분류함[cite: 3].
   - **BMI 및 비만 유병:** 자기보고 신장(`HT`)과 체중(`WT`) 데이터 기반 $\text{체중}(\text{kg}) / (\text{신장}(\text{m}))^2$ 산출 후, BMI $\ge 25.0 \text{ kg/m}^2$를 비만 유병으로 정의함[cite: 3].
4. **결측 처리:** 공변량 결측으로 인한 표본 손실 및 편향을 방지하기 위해 일괄 제거(Listwise Deletion)를 하지 않고 변수별 유효 응답을 기준으로 가중 비율을 추정함[cite: 1, 3].
5. **결과 해석 주의 (Causal Inference Warning):** 본 분석에서 제시된 아침 결식군과 비결식군 간의 비만 유병률 차이는 단일 시점에서 수집된 횡단면 자료(Cross-sectional Data)의 관찰된 관련성(Association)을 의미하며, 인과적 효과(Causality)를 직접적으로 입증하지 않음.
6. **변수 보정 관련 주의 (Weight Control Variable):** 체중 조절 시도(`WC_MN`)는 결식 행위의 원인이 아니라 비만 판정 이후 선행된 행동 변화일 가능성(역인과성/매개변수/동반선택 편향 등)이 존재함[cite: 2, 3]. 따라서 주 분석 모형에 자동 보정변수로 포함하지 않으며, 11주차 DAG(인과이행도) 설정 및 12주차 민감도 분석(Sensitivity Analysis)을 통해 인과적 역할을 다각도로 검토함[cite: 2].